In [ ]:
transformer_decrease_trends = spark.sql("""
with hourlymetrics as (
    select 
        transformer_id,
        hour(hour_ts) as hour_of_day,
        count(*) as no_of_decrease,
        round(avg(percent_change), 2) as avg_impact
    from `electric_analysis_dev`.`transformer_hourly_analysis`
    where load_change = 'Decrease'
    group by 1, 2
),
rankedpeaks as (
    select 
        *,
        row_number() over (
            partition by transformer_id 
            order by no_of_decrease desc
        ) as rnk
    from hourlymetrics
)
select 
    transformer_id,
    hour_of_day as drop_hour,
    no_of_decrease as days_with_decrease,
    avg_impact
from rankedpeaks
where rnk = 1
""")

spark.sql("create database if not exists `electric_analysis_dev`")

transformer_decrease_trends.coalesce(1).write \
    .mode("overwrite") \
    .format("parquet") \
    .option("path", "s3://ops-autopilot-data/transformed/transformer_decrease_trends/") \
    .saveAsTable("`electric_analysis_dev`.`transformer_decrease_trends`")